# Is the Chain-of-Thought Actually Thinking? — Results

CS 639 Spring 2026. Run this *after* all four scripts have completed:

```
python -m scripts.make_subset
python -m scripts.run_baseline
python -m scripts.run_perturbations
python -m scripts.run_attention --store-full
python -m scripts.analyze
```

In [ ]:
import json, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.evaluate import build_summary, by_difficulty
from src import viz

BASE = pathlib.Path('results/baseline_cots.jsonl')
PERT = pathlib.Path('results/perturbed_cots.jsonl')
ATTN = pathlib.Path('results/attention')
FIG = pathlib.Path('results/figures'); FIG.mkdir(parents=True, exist_ok=True)

## 1. Load data

In [ ]:
base = pd.read_json(BASE, lines=True)
pert = pd.read_json(PERT, lines=True)
print(f'Baseline: {len(base)} rows across {base["model"].nunique()} models')
print(f'Perturbed: {len(pert)} rows')
base.head()

## 2. Baseline accuracy

In [ ]:
acc = base.groupby(['model','dataset'])['correct'].mean().unstack()
print(acc.round(3))
acc.plot(kind='bar', figsize=(8,4), ylim=(0,1)); plt.ylabel('accuracy'); plt.title('Baseline accuracy');

## 3. Faithfulness score F (macro)

In [ ]:
summary = build_summary(base, pert)
summary.round(3)

In [ ]:
for ds in summary['dataset'].unique():
    viz.plot_faithfulness_heatmap(summary, ds, FIG / f'faithfulness_{ds}.png')
    from IPython.display import Image, display
    display(Image(FIG / f'faithfulness_{ds}.png'))

## 4. Hypothesis 1: faithfulness vs. difficulty

In [ ]:
diff_df = by_difficulty(base, pert)
viz.plot_f_by_difficulty(diff_df, FIG / 'f_by_difficulty.png')
from IPython.display import Image
Image(FIG / 'f_by_difficulty.png')

## 5. Hypothesis 2: model size comparison

Averaged across perturbations and datasets.

In [ ]:
by_model = summary.groupby('model')['F'].mean().sort_values()
print(by_model.round(3))
by_model.plot(kind='barh', figsize=(6,3)); plt.xlabel('Mean F'); plt.title('Faithfulness by model');

## 6. Hypothesis 3: attention ↔ behavior convergence (Llama)

In [ ]:
scatter_path = ATTN / 'scatter.json'
scatter_df = pd.DataFrame(json.loads(scatter_path.read_text())) if scatter_path.exists() else pd.DataFrame()
if len(scatter_df):
    viz.plot_convergence(scatter_df, FIG / 'convergence_scatter.png')
    from IPython.display import Image
    display(Image(FIG / 'convergence_scatter.png'))
    scatter_df.groupby('changed_answer')['cot_attention_ratio'].describe()

## 7. Micro case studies

In [ ]:
from IPython.display import Image, display
for p in sorted(FIG.glob('micro_*.png'))[:3]:
    display(Image(p))

## 8. Attention heatmaps (faithful vs unfaithful pair)

In [ ]:
from IPython.display import Image, display
for p in sorted(FIG.glob('attention_*.png'))[:2]:
    display(Image(p))